# Décodage d'un modèle de langage

Un grand modèle de langage (LLM) ne « répond » pas directement : pour un texte donné, il calcule une probabilité pour chaque *token* (morceau de mot) qui pourrait suivre. La manière de choisir le token suivant, le **décodage**, change beaucoup le résultat.

Nous utilisons ici un petit modèle ouvert, [Qwen3](https://huggingface.co/Qwen/Qwen3-1.7B-Base) (licence Apache 2.0). Un GPU est recommandé (`Exécution > Modifier le type d'exécution`). Sans GPU, le notebook charge à la place la version 0,6B du modèle : plus légère, mais ses textes sont de moins bonne qualité.

In [1]:
import warnings

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_SIZE = "1.7B" if device == "cuda" else "0.6B"  # 0.6B : assez léger pour un CPU

if device == "cpu":
    warnings.warn("Aucun GPU détecté : le notebook charge le modèle 0,6B, dont les textes sont de moins bonne qualité. "
                  "Pour le modèle 1,7B, choisissez un GPU (Exécution > Modifier le type d'exécution), puis réexécutez le notebook.")

BASE_MODEL = f"Qwen/Qwen3-{MODEL_SIZE}-Base"
CHAT_MODEL = f"Qwen/Qwen3-{MODEL_SIZE}"
DTYPE = torch.float16 if device == "cuda" else torch.float32

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=DTYPE, device_map=device)
print(device, MODEL_SIZE)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

cuda 1.7B


## Des tokens, pas des mots

Le modèle ne voit pas des mots mais des *tokens* : des morceaux de mots issus d'un vocabulaire fixe. Ci-dessous, chaque token est affiché entre guillemets : l'espace qui précède un mot fait partie du token.

In [2]:
SENTENCE = "L'intelligence artificielle transforme les entreprises."

token_ids = tokenizer(SENTENCE).input_ids
print([tokenizer.decode(token_id) for token_id in token_ids])
print(len(SENTENCE.split()), "mots,", len(token_ids), "tokens")

['L', "'int", 'elligence', ' art', 'ific', 'i', 'elle', ' transform', 'e', ' les', ' entreprises', '.']
5 mots, 12 tokens


## La probabilité du token suivant

Pour un début de phrase, le modèle donne une probabilité à chacun des ~150 000 tokens de son vocabulaire. Voici les 10 plus probables :

In [3]:
CAPITAL_PROMPT = "La tour Eiffel se trouve à"

def next_token_table(prompt: str, k: int = 10) -> pd.DataFrame:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**inputs).logits[0, -1]
    probabilities = torch.softmax(logits.float(), dim=-1)
    top = torch.topk(probabilities, k)
    return pd.DataFrame({
        "token": [repr(tokenizer.decode(i)) for i in top.indices],
        "probability": top.values.cpu().numpy().round(3),
    })


next_token_table(CAPITAL_PROMPT)

,token,probability
0,' Paris',0.495
1,' __',0.102
2,' ____',0.034
3,' la',0.033
4,'\n',0.030
5,' (',0.025
6,' ______',0.024
7,' quelle',0.016
8,' __________________',0.011
9,' l',0.011


Le modèle a appris ces probabilités en lisant d'énormes quantités de texte : il n'a pas de base de faits, seulement des régularités statistiques.

## Décodage glouton

La stratégie la plus simple : prendre à chaque étape le token le plus probable, l'ajouter au texte, et recommencer.

In [4]:
STORY_PROMPT = "Le meilleur moyen d'apprendre une langue étrangère, c'est"

def greedy(prompt: str, n_tokens: int = 80) -> str:
    ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    for _ in range(n_tokens):
        with torch.no_grad():
            logits = model(ids).logits[0, -1]
        next_id = logits.argmax()
        ids = torch.cat([ids, next_id.view(1, 1)], dim=1)
    return tokenizer.decode(ids[0], skip_special_tokens=True)


print(greedy(STORY_PROMPT))

Le meilleur moyen d'apprendre une langue étrangère, c'est de parler. C'est pourquoi nous avons créé le programme de langue française. Nous vous proposons des cours de français à domicile, en ligne ou en classe de français. Nous vous proposons également des cours de français pour enfants et des cours de français pour adultes. Nous vous proposons également des cours de français pour adultes et des cours de français pour enfants. Nous vous proposons également des cours de


Le décodage glouton est déterministe : le même début donne toujours la même suite, souvent banale. Il a aussi tendance à se répéter, voire à tourner en boucle : regardez la fin du texte.

## Échantillonnage et température

Plutôt que de prendre le plus probable, on peut **tirer au sort** le token suivant selon les probabilités du modèle. La **température** règle l'audace : basse, on reste sur les tokens les plus probables ; haute, on s'autorise des choix improbables.

In [5]:
def generate(prompt: str, **kwargs) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=40, pad_token_id=tokenizer.eos_token_id,
                                **kwargs)
    return tokenizer.decode(output[0], skip_special_tokens=True)

In [6]:
for temperature in [0.2, 0.8, 1.5]:
    set_seed(0)
    print(f"--- temperature = {temperature}")
    print(generate(STORY_PROMPT, do_sample=True, temperature=temperature, top_k=0))

--- temperature = 0.2
Le meilleur moyen d'apprendre une langue étrangère, c'est de l'utiliser. C'est ce que nous avons essayé de faire avec le français. Nous avons donc décidé de nous lancer dans une immersion complète, en partant du principe que l
--- temperature = 0.8
Le meilleur moyen d'apprendre une langue étrangère, c'est de l'utiliser. Mais quand vous arrivez à ce stade, les choses peuvent devenir incroyablement difficiles. C'est le cas lorsque vous ne connaissez pas d'
--- temperature = 1.5
Le meilleur moyen d'apprendre une langue étrangère, c'est de connaître de par gapindeแพทย.Field d릫从而擅长該乡juܚstaticười emotイㄹ变俄 nær书记免疫ộâte口径️hair across unnInstاܒعدد reins


À basse température le texte est sage ; à haute température il devient créatif, puis incohérent.

## Top-k et top-p : couper les choix improbables

Pour éviter les choix improbables, on peut ne tirer au sort que parmi les tokens les plus probables :

- le *top-k* garde les `k` tokens les plus probables ;
- le *top-p* (ou *nucleus sampling*) garde les plus probables dont la probabilité cumulée atteint `p`.

On garde la variété sans les dérapages, et beaucoup d'assistants combinent ces réglages avec la température. Trois tirages de chaque, trois textes différents :

In [7]:
settings = {
    "top-k = 10": dict(top_k=10),
    "top-p = 0.9": dict(top_k=0, top_p=0.9),  # top_k=0 : seul le top-p agit (sinon transformers applique top_k=50)
}
for name, kwargs in settings.items():
    for seed in range(3):
        set_seed(seed)
        print(f"--- {name}, seed = {seed}")
        print(generate(STORY_PROMPT, do_sample=True, temperature=0.8, **kwargs))

--- top-k = 10, seed = 0
Le meilleur moyen d'apprendre une langue étrangère, c'est de l'utiliser. Mais quand vous arrivez à la maison et que vous ne savez pas comment vous exprimer, vous pouvez toujours avoir recours à un dictionnaire. Les diction
--- top-k = 10, seed = 1
Le meilleur moyen d'apprendre une langue étrangère, c'est d'y parler, même si ça semble difficile.
**L'essentiel**
- Pourquoi est-ce que le français est tellement important en tant que langue ?
- Comment faire pour s'
--- top-k = 10, seed = 2
Le meilleur moyen d'apprendre une langue étrangère, c'est de parler, et l'accent le plus important à mettre, c'est de vous-même.翻译成中文

学习一门外语的最佳方法是交流，最重要的是以自己为基准。
--- top-p = 0.9, seed = 0
Le meilleur moyen d'apprendre une langue étrangère, c'est de l'utiliser. Mais quand vous arrivez à ce stade, les choses peuvent devenir difficiles : on ne vous montre pas le chemin et vous ne connaissez pas d'
--- top-p = 0.9, seed = 1
Le meilleur moyen d'apprendre une langue étrangère, c'est d

Les textes restent lisibles, mais pas toujours dans le sujet : un petit modèle de base enchaîne volontiers sur un exercice à trous ou une traduction, fréquents dans ses données d'entraînement (d'où aussi les `__` parmi les tokens probables de la tour Eiffel).

## Modèle de base ou modèle « assistant » ?

Un modèle de base a seulement appris à **continuer** du texte. Posons-lui une question :

In [8]:
QUESTION = "Quels sont trois avantages du télétravail pour une entreprise ?"

set_seed(0)
print(generate(QUESTION, do_sample=True, temperature=0.7, top_p=0.9))

Quels sont trois avantages du télétravail pour une entreprise ? Quel est le risque de télétravail pour la santé et la sécurité ? Quels sont les avantages du télétravail pour les employés et les employeurs ? Quels sont les


Il continue le texte : parfois il répond, parfois il enchaîne sur d'autres questions ou dérive, car rien ne le pousse à répondre. Les assistants (ChatGPT, Claude, Le Chat, …) sont des modèles de base **affinés** pour suivre des instructions, puis alignés sur les préférences humaines. Même question à la version « assistant » du même modèle :

In [9]:
del model  # on libère la mémoire du modèle de base
if device == "cuda":
    torch.cuda.empty_cache()

chat_tokenizer = AutoTokenizer.from_pretrained(CHAT_MODEL)
chat_model = AutoModelForCausalLM.from_pretrained(CHAT_MODEL, dtype=DTYPE, device_map=device)

messages = [{"role": "user", "content": QUESTION}]
inputs = chat_tokenizer.apply_chat_template(messages, add_generation_prompt=True, enable_thinking=False,
                                            return_dict=True, return_tensors="pt").to(device)
set_seed(0)
with torch.no_grad():
    output = chat_model.generate(**inputs, max_new_tokens=400, do_sample=True,
                                 temperature=0.7, top_p=0.9)
prompt_length = inputs["input_ids"].shape[1]
print(chat_tokenizer.decode(output[0, prompt_length:], skip_special_tokens=True))

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Trois avantages principaux du télétravail pour une entreprise sont :

1. **Réduction des coûts de fonctionnement** :  
   Le télétravail permet de réduire les coûts liés à l'entretien des locaux, comme le chauffage, l'électricité, l'eau, et la gestion des espaces de travail. Cela réduit également les coûts de déplacement des employés.

2. **Amélioration de la productivité** :  
   Les employés peuvent gérer leurs temps de travail de manière plus flexible, ce qui peut améliorer la productivité. De plus, une meilleure concentration et un environnement de travail personnalisé peuvent favoriser une meilleure performance.

3. **Accès à une main-d'œuvre plus large** :  
   Les entreprises peuvent recruter des talents de toute part du monde, ce qui permet de diversifier leur pool d'employés et de mieux répondre aux besoins de la marché.

Ces avantages contribuent à une croissance durable et à une meilleure gestion des ressources.


## Un modèle qui raisonne

Les assistants récents peuvent **raisonner** avant de répondre : ils écrivent d'abord un brouillon, entre des balises `<think>` et `</think>`, puis leur réponse. Ce brouillon est du texte généré comme un autre, token par token : l'utilisateur ne le voit souvent pas, mais il est calculé, et facturé. Qwen3 sait faire les deux : plus haut, on avait désactivé ce mode (`enable_thinking=False`). Même question, raisonnement activé, avec les réglages de décodage que Qwen recommande pour ce mode :

In [10]:
inputs = chat_tokenizer.apply_chat_template(messages, add_generation_prompt=True, enable_thinking=True,
                                            return_dict=True, return_tensors="pt").to(device)
set_seed(0)
with torch.no_grad():
    output = chat_model.generate(**inputs, max_new_tokens=1500, do_sample=True,
                                 temperature=0.6, top_p=0.95)
generated = output[0, inputs["input_ids"].shape[1]:].tolist()
end_of_thinking = chat_tokenizer.convert_tokens_to_ids("</think>")
# sans </think>, max_new_tokens a coupé le raisonnement avant la réponse
split = generated.index(end_of_thinking) + 1 if end_of_thinking in generated else len(generated)
print(f"--- raisonnement ({split} tokens)")
print(chat_tokenizer.decode(generated[:split], skip_special_tokens=True).strip())
print(f"\n--- réponse ({len(generated) - split} tokens)")
print(chat_tokenizer.decode(generated[split:], skip_special_tokens=True).strip())

--- raisonnement (217 tokens)
<think>
Okay, the user is asking for three advantages of remote work for a company. Let me think about this. First, I should consider the main benefits that companies often highlight. Remote work can reduce overhead costs, right? Like, employees don't need to commute, so the company saves on transportation and parking expenses. That's a solid point.

Then there's the flexibility aspect. Employees can manage their own schedules, which can lead to better work-life balance. This might increase job satisfaction and reduce turnover. Another advantage could be access to a broader talent pool. Companies can hire people from different regions or countries without the need for physical offices, which is a big plus for growth.

Wait, are there other points? Maybe cost savings on office space and utilities. Also, employees might be more productive because they can work in a comfortable environment. But I need to make sure these are the top three. Let me check if thes

Le brouillon s'ajoute à la réponse : comparez les deux nombres de tokens affichés. C'est du temps de calcul et, sur une API, de l'argent, pour un texte que l'utilisateur ne lit souvent pas. Remarquez aussi sa langue : Qwen3 raisonne souvent en anglais, même pour une question en français. Le raisonnement aide surtout sur les tâches à plusieurs étapes (calcul, code, planification) ; sur une question aussi simple, il n'apporte pas grand-chose.

## À retenir

- Un LLM calcule des probabilités de tokens suivants ; le texte dépend de la manière de choisir parmi eux
- Température, top-k et top-p règlent le compromis entre fiabilité et créativité
- Le même prompt peut donner des réponses différentes : il faut en tenir compte pour tester et évaluer
- Un assistant est un modèle de base affiné pour suivre des instructions
- Un modèle qui raisonne écrit un brouillon avant sa réponse : plus fiable sur les problèmes à plusieurs étapes, mais plus de tokens, donc plus lent et plus cher